In [4]:
# import joblib

# saved = joblib.load(save_dir / "research_state.joblib")

In [5]:
# Loading correct experiment data
from pathlib import Path
from collections import defaultdict, deque

import joblib
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()

checkpoint_dir = (
    PROJECT_ROOT
    / "checkpoints"
    / "v1_1_corrected_20260915_190534_952693"
)

experiment = joblib.load(
    checkpoint_dir / "experiment_data.joblib"
)
historical_bundle = joblib.load(
    checkpoint_dir / "model_bundle.joblib"
)

matches = (
    experiment["df"]
    .copy()
    .sort_values("date", kind="stable")
    .reset_index(drop=True)
)

baseline_features = list(historical_bundle["features"])

In [6]:
# Calculate defense and opponent elo context from each historical match
from collections import defaultdict, deque
import numpy as np
import pandas as pd

matches = (
    matches.sort_values("date", kind="stable")
    .reset_index(drop=True)
    .copy()
)

required_columns = [
    "date",
    "home_team",
    "away_team",
    "home_score",
    "away_score",
    "home_elo_pre_match",
    "away_elo_pre_match",
]

assert matches[required_columns].notna().all().all()
assert matches["tournament"].ne("Friendly").all()

# Each entry contains:
# (goals conceded, opponent's pre-match Elo)
recent_history = defaultdict(lambda: deque(maxlen=5))

new_columns = [
    "home_goals_conceded_5",
    "away_goals_conceded_5",
    "home_opponent_elo_5",
    "away_opponent_elo_5",
]

values = {
    column: np.full(len(matches), np.nan)
    for column in new_columns
}

for date, daily_matches in matches.groupby("date", sort=False):
    # Extract features before recording ANY results from this date.
    for index, match in daily_matches.iterrows():
        for side in ["home", "away"]:
            team = match[f"{side}_team"]
            past = recent_history[team]

            if past:
                values[f"{side}_goals_conceded_5"][index] = (
                    np.mean([entry[0] for entry in past])
                )

                values[f"{side}_opponent_elo_5"][index] = (
                    np.mean([entry[1] for entry in past])
                )

    # Update history after extracting all this date's features.
    for _, match in daily_matches.iterrows():
        recent_history[match["home_team"]].append((
            float(match["away_score"]),
            float(match["away_elo_pre_match"]),
        ))

        recent_history[match["away_team"]].append((
            float(match["home_score"]),
            float(match["home_elo_pre_match"]),
        ))

for column, array in values.items():
    matches[column] = array

display(matches[
    ["date", "home_team", "away_team"] + new_columns
].tail(10))

,date,home_team,away_team,home_goals_conceded_5,away_goals_conceded_5,home_opponent_elo_5,away_opponent_elo_5
31079,2026-06-26,Cape Verde,Saudi Arabia,1.6,1.6,1834.635159,1921.012703
31080,2026-06-26,Uruguay,Spain,0.6,0.4,1744.813395,1684.517435
31081,2026-06-26,Norway,France,1.0,0.8,1700.983608,1658.511221
31082,2026-06-26,Senegal,Iraq,1.8,2.2,1915.104796,1908.152134
31083,2026-06-27,Algeria,Austria,1.4,1.0,1802.343571,1718.903515
31084,2026-06-27,Jordan,Argentina,1.6,0.4,1833.725485,1877.207125
31085,2026-06-27,Colombia,Portugal,1.0,1.2,1876.727025,1684.318152
31086,2026-06-27,DR Congo,Uzbekistan,0.6,1.8,1817.802560,1847.773679
31087,2026-06-27,Panama,England,1.0,0.4,1698.326167,1690.225655
31088,2026-06-27,Croatia,Ghana,1.4,0.4,1562.514594,1772.089034


In [7]:
# Scoring Level, scoring difference
matches["scoring_level"] = (
    matches["home_rolling_goals"]
    + matches["away_rolling_goals"]
)

development = (
    matches.loc[matches["date"] < "2018-01-01"]
    .reset_index(drop=True)
    .copy()
)

# Confirm the saved fold indices still refer to the same matches.
pd.testing.assert_frame_equal(
    development[["date", "home_team", "away_team"]],
    experiment["development"]
        .reset_index(drop=True)[["date", "home_team", "away_team"]],
)

folds = experiment["folds"]

current_features = (
    list(baseline_features)
    + [
        "home_goals_conceded_5",
        "away_goals_conceded_5",
        "home_opponent_elo_5",
        "away_opponent_elo_5",
    ]
)

feature_sets = {
    "current": current_features,
    "current_plus_scoring": current_features + ["scoring_level"],
}

In [8]:
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import log_loss

def compare_feature_sets(frame, folds, feature_sets, model_templates):
    records = []

    for model_name, template in model_templates.items():
        for feature_name, columns in feature_sets.items():
            X = frame[columns].astype(float)
            y = frame["target"]

            for fold_number, (train_idx, valid_idx) in enumerate(
                folds, start=1
            ):
                estimator = Pipeline([
                    ("imputer", SimpleImputer(
                        strategy="median",
                        add_indicator=True,
                    )),
                    ("model", clone(template)),
                ])

                estimator.fit(
                    X.iloc[train_idx],
                    y.iloc[train_idx],
                )

                assert np.array_equal(estimator.classes_, [0, 1, 2])

                probabilities = estimator.predict_proba(
                    X.iloc[valid_idx]
                )

                records.append({
                    "model": model_name,
                    "feature_set": feature_name,
                    "fold": fold_number,
                    "log_loss": log_loss(
                        y.iloc[valid_idx],
                        probabilities,
                        labels=[0, 1, 2],
                    ),
                })

    return pd.DataFrame(records)

In [9]:
# Reuse the development partition and folds
development = (
    matches.loc[matches["date"] < "2018-01-01"]
    .reset_index(drop=True)
    .copy()
)

original_development = (
    experiment["development"]
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(
    development[["date", "home_team", "away_team"]],
    original_development[["date", "home_team", "away_team"]],
)

folds = experiment["folds"]

In [10]:
# Compare baseline LR against LR plus defensive features
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import log_loss

historical_lr_pipeline = (
    historical_bundle["model"].estimator.estimator
)

defense_features = [
    "home_goals_conceded_5",
    "away_goals_conceded_5",
]

opponent_features = [
    "home_opponent_elo_5",
    "away_opponent_elo_5",
]

feature_sets = {
    "baseline": list(baseline_features),

    "baseline_plus_defense": (
        list(baseline_features)
        + defense_features
    ),

    "defense_plus_opponent_context": (
        list(baseline_features)
        + defense_features
        + opponent_features
    ),
}

records = []

for feature_set, columns in feature_sets.items():
    X = development[columns].astype(float)
    y = development["target"]

    for fold_number, (train_idx, valid_idx) in enumerate(folds, start=1):
        model = Pipeline([
            ("imputer", SimpleImputer(
                strategy="median",
                add_indicator=True,
            )),
            ("lr_pipeline", clone(historical_lr_pipeline)),
        ])

        model.fit(X.iloc[train_idx], y.iloc[train_idx])

        probabilities = model.predict_proba(X.iloc[valid_idx])

        records.append({
            "feature_set": feature_set,
            "fold": fold_number,
            "log_loss": log_loss(
                y.iloc[valid_idx],
                probabilities,
                labels=[0, 1, 2],
            ),
        })

defense_results = pd.DataFrame(records)

comparison = defense_results.pivot(
    index="fold",
    columns="feature_set",
    values="log_loss",
)

comparison["delta_defense_minus_baseline"] = (
    comparison["baseline_plus_defense"]
    - comparison["baseline"]
)

display(comparison.round(6))
print("\nMean results:")
print(comparison.mean().round(6))

feature_set,baseline,baseline_plus_defense,defense_plus_opponent_context,delta_defense_minus_baseline
fold,,,,
1,0.863823,0.855082,0.852920,-0.008741
2,0.875552,0.868663,0.868797,-0.006889
3,0.909705,0.901248,0.899183,-0.008457
4,0.873515,0.870795,0.868673,-0.002720



Mean results:
feature_set
baseline                         0.880649
baseline_plus_defense            0.873947
defense_plus_opponent_context    0.872393
delta_defense_minus_baseline    -0.006702
dtype: float64


In [11]:
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import log_loss
from xgboost import XGBClassifier

model_templates = {
    "LR": clone(historical_lr_pipeline),

    "XGBoost": XGBClassifier(
        objective="multi:softprob",
        eval_metric="mlogloss",
        max_depth=3,
        min_child_weight=1,
        reg_lambda=1.0,
        learning_rate=0.05,
        n_estimators=150,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=1,
    ),
}

records = []

for model_name, template in model_templates.items():
    for feature_set_name, columns in feature_sets.items():
        X = development[columns].astype(float)
        y = development["target"]

        for fold_number, (train_idx, valid_idx) in enumerate(
            folds, start=1
        ):
            estimator = Pipeline([
                ("imputer", SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                )),
                ("estimator", clone(template)),
            ])

            estimator.fit(
                X.iloc[train_idx],
                y.iloc[train_idx],
            )

            assert np.array_equal(
                estimator.classes_, [0, 1, 2]
            )

            probabilities = estimator.predict_proba(
                X.iloc[valid_idx]
            )

            records.append({
                "model": model_name,
                "feature_set": feature_set_name,
                "fold": fold_number,
                "log_loss": log_loss(
                    y.iloc[valid_idx],
                    probabilities,
                    labels=[0, 1, 2],
                ),
            })

model_defense_results = pd.DataFrame(records)
fold_comparison = model_defense_results.pivot(
    index=["model", "fold"],
    columns="feature_set",
    values="log_loss",
)

# Isolate the contribution of the NEW opponent features.
fold_comparison["delta_opponent_context"] = (
    fold_comparison["defense_plus_opponent_context"]
    - fold_comparison["baseline_plus_defense"]
)

display(fold_comparison.round(6))

summary = (
    fold_comparison.reset_index()
    .groupby("model")
    .agg(
        baseline_loss=("baseline", "mean"),
        defense_loss=("baseline_plus_defense", "mean"),
        defense_opponent_loss=(
            "defense_plus_opponent_context", "mean"
        ),
        opponent_context_delta=("delta_opponent_context", "mean"),
        folds_improved=(
            "delta_opponent_context",
            lambda values: int((values < 0).sum()),
        ),
    )
)

display(summary.round(6))

feature_set   baseline  baseline_plus_defense  defense_plus_opponent_context  \
model   fold                                                                   
LR      1     0.863823               0.855082                       0.852920   
        2     0.875552               0.868663                       0.868797   
        3     0.909705               0.901248                       0.899183   
        4     0.873515               0.870795                       0.868673   
XGBoost 1     0.864906               0.857901                       0.856671   
        2     0.875656               0.867196                       0.870384   
        3     0.910671               0.900793                       0.899103   
        4     0.873511               0.869772                       0.867930   

feature_set   delta_opponent_context  
model   fold                          
LR      1                  -0.002162  
        2                   0.000134  
        3                  -0.002065  
        4                  -0.002122  
XGBoost 1                  -0.001231  
        2                   0.003188  
        3                  -0.001690  
        4                  -0.001842

,baseline_loss,defense_loss,defense_opponent_loss,opponent_context_delta,folds_improved
model,,,,,
LR,0.880649,0.873947,0.872393,-0.001554,3
XGBoost,0.881186,0.873915,0.873522,-0.000394,3


In [12]:
# Dedicated feature lists for the scoring experiment.
scoring_base_features = list(dict.fromkeys(
    list(baseline_features) + [
        "home_goals_conceded_5",
        "away_goals_conceded_5",
        "home_opponent_elo_5",
        "away_opponent_elo_5",
    ]
))

development = development.copy()
development["scoring_level"] = (
    development["home_rolling_goals"]
    + development["away_rolling_goals"]
)

scoring_feature_sets = {
    "current": scoring_base_features,
    "current_plus_scoring": scoring_base_features + ["scoring_level"],
}

scoring_results = compare_feature_sets(
    development,
    folds,
    scoring_feature_sets,
    model_templates,
)

scoring_comparison = scoring_results.pivot(
    index=["model", "fold"],
    columns="feature_set",
    values="log_loss",
)

scoring_comparison["delta"] = (
    scoring_comparison["current_plus_scoring"]
    - scoring_comparison["current"]
)

scoring_summary = scoring_comparison.groupby("model").agg(
    current_loss=("current", "mean"),
    scoring_loss=("current_plus_scoring", "mean"),
    mean_delta=("delta", "mean"),
    folds_improved=("delta", lambda values: (values < 0).sum()),
)

display(scoring_comparison)
display(scoring_summary)

feature_set    current  current_plus_scoring     delta
model   fold                                          
LR      1     0.852920              0.852064 -0.000856
        2     0.868797              0.868471 -0.000326
        3     0.899183              0.899641  0.000458
        4     0.868673              0.868382 -0.000291
XGBoost 1     0.856671              0.855892 -0.000778
        2     0.870384              0.869935 -0.000449
        3     0.899103              0.897919 -0.001184
        4     0.867930              0.868317  0.000388

,current_loss,scoring_loss,mean_delta,folds_improved
model,,,,
LR,0.872393,0.872140,-0.000254,3
XGBoost,0.873522,0.873016,-0.000506,3


In [13]:
# Recency weighting implementation (feature)
from collections import defaultdict, deque

import numpy as np
import pandas as pd

matches = matches.copy()
matches["date"] = pd.to_datetime(matches["date"])

matches = (
    matches.sort_values("date", kind="stable")
    .reset_index(drop=True)
)

required = [
    "date",
    "home_team",
    "away_team",
    "home_score",
    "away_score",
    "home_elo_pre_match",
    "away_elo_pre_match",
]

assert matches[required].notna().all().all()
assert matches["tournament"].ne("Friendly").all()

# None means equal weighting.
weighting_options = {
    "equal": None,
    "180d": 180,
    "365d": 365,
}

# Each entry: (date, goals conceded, opponent pre-match Elo)
team_history = defaultdict(lambda: deque(maxlen=5))

generated_columns = {}

for suffix in weighting_options:
    for side in ["home", "away"]:
        for measure in ["conceded", "opponent_elo"]:
            column = f"{side}_{measure}_5_{suffix}"
            generated_columns[column] = np.full(len(matches), np.nan)


def summarize_history(entries, prediction_date, half_life_days):
    if not entries:
        return np.nan, np.nan

    dates = [entry[0] for entry in entries]
    conceded = np.array([entry[1] for entry in entries], dtype=float)
    opponent_elos = np.array([entry[2] for entry in entries], dtype=float)

    ages = np.array([
        (prediction_date - date).days
        for date in dates
    ], dtype=float)

    assert (ages > 0).all(), "History must come from earlier dates."

    if half_life_days is None:
        weights = np.ones(len(entries))
    else:
        # Subtracting the youngest age avoids numerical underflow.
        # It does not change the normalized weighted average.
        weights = np.exp2(
            -(ages - ages.min()) / half_life_days
        )

    return (
        float(np.average(conceded, weights=weights)),
        float(np.average(opponent_elos, weights=weights)),
    )


for date, daily_matches in matches.groupby("date", sort=False):
    # First generate features using only previous dates.
    for index, match in daily_matches.iterrows():
        for side in ["home", "away"]:
            entries = team_history[match[f"{side}_team"]]

            for suffix, half_life in weighting_options.items():
                conceded_avg, opponent_avg = summarize_history(
                    entries,
                    date,
                    half_life,
                )

                generated_columns[
                    f"{side}_conceded_5_{suffix}"
                ][index] = conceded_avg

                generated_columns[
                    f"{side}_opponent_elo_5_{suffix}"
                ][index] = opponent_avg

    # Only after generating the day's features, record its results.
    for _, match in daily_matches.iterrows():
        team_history[match["home_team"]].append((
            date,
            float(match["away_score"]),
            float(match["away_elo_pre_match"]),
        ))

        team_history[match["away_team"]].append((
            date,
            float(match["home_score"]),
            float(match["home_elo_pre_match"]),
        ))

for column, values in generated_columns.items():
    matches[column] = values

print("Generated equal-weight, 180-day, and 365-day features.")

Generated equal-weight, 180-day, and 365-day features.


In [14]:
equivalent_columns = {
    "home_goals_conceded_5": "home_conceded_5_equal",
    "away_goals_conceded_5": "away_conceded_5_equal",
    "home_opponent_elo_5": "home_opponent_elo_5_equal",
    "away_opponent_elo_5": "away_opponent_elo_5_equal",
}

for original, rebuilt in equivalent_columns.items():
    if original in matches.columns:
        np.testing.assert_allclose(
            matches[original].to_numpy(dtype=float),
            matches[rebuilt].to_numpy(dtype=float),
            equal_nan=True,
            err_msg=f"Reference feature changed: {original}",
        )

print("Available reference columns checked.")

Available reference columns checked.


In [15]:
INCLUDE_SCORING_LEVEL = False

common_features = list(baseline_features)

if INCLUDE_SCORING_LEVEL:
    matches["scoring_level"] = (
        matches["home_rolling_goals"]
        + matches["away_rolling_goals"]
    )
    common_features.append("scoring_level")

recency_feature_sets = {}

for suffix in weighting_options:
    recency_feature_sets[suffix] = common_features + [
        f"home_conceded_5_{suffix}",
        f"away_conceded_5_{suffix}",
        f"home_opponent_elo_5_{suffix}",
        f"away_opponent_elo_5_{suffix}",
    ]

development = (
    matches.loc[matches["date"] < "2018-01-01"]
    .reset_index(drop=True)
    .copy()
)

original_development = (
    experiment["development"]
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(
    development[["date", "home_team", "away_team"]],
    original_development[["date", "home_team", "away_team"]],
)

folds = experiment["folds"]

In [16]:
recency_results = compare_feature_sets(
    development,
    folds,
    recency_feature_sets,
    model_templates,
)

recency_comparison = recency_results.pivot(
    index=["model", "fold"],
    columns="feature_set",
    values="log_loss",
)

recency_comparison["delta_180d"] = (
    recency_comparison["180d"]
    - recency_comparison["equal"]
)

recency_comparison["delta_365d"] = (
    recency_comparison["365d"]
    - recency_comparison["equal"]
)

display(recency_comparison.round(6))

recency_summary = (
    recency_comparison.reset_index()
    .groupby("model")
    .agg(
        equal_loss=("equal", "mean"),
        loss_180d=("180d", "mean"),
        loss_365d=("365d", "mean"),
        mean_delta_180d=("delta_180d", "mean"),
        mean_delta_365d=("delta_365d", "mean"),
        folds_improved_180d=(
            "delta_180d",
            lambda values: int((values < 0).sum()),
        ),
        folds_improved_365d=(
            "delta_365d",
            lambda values: int((values < 0).sum()),
        ),
    )
)

display(recency_summary.round(6))

feature_set       180d      365d     equal  delta_180d  delta_365d
model   fold                                                      
LR      1     0.854557  0.853956  0.852920    0.001637    0.001036
        2     0.869795  0.870254  0.868797    0.000997    0.001457
        3     0.897846  0.898189  0.899183   -0.001337   -0.000994
        4     0.868218  0.868419  0.868673   -0.000455   -0.000254
XGBoost 1     0.858608  0.857802  0.856671    0.001937    0.001131
        2     0.869662  0.867507  0.870384   -0.000722   -0.002878
        3     0.896642  0.896983  0.899103   -0.002461   -0.002120
        4     0.868181  0.868216  0.867930    0.000251    0.000287

,equal_loss,loss_180d,loss_365d,mean_delta_180d,mean_delta_365d,folds_improved_180d,folds_improved_365d
model,,,,,,,
LR,0.872393,0.872604,0.872705,0.000211,0.000311,2,2
XGBoost,0.873522,0.873273,0.872627,-0.000249,-0.000895,2,2


In [17]:
# Preparing combined feature sets for recency experiment with scoring level
import pandas as pd

combined_development = (
    matches.loc[pd.to_datetime(matches["date"]) < "2018-01-01"]
    .copy()
    .reset_index(drop=True)
)

# Ensure your existing fold indices still identify the same matches.
identity_columns = ["date", "home_team", "away_team"]

pd.testing.assert_frame_equal(
    combined_development[identity_columns],
    development.reset_index(drop=True)[identity_columns],
)

combined_development["scoring_level"] = (
    combined_development["home_rolling_goals"]
    + combined_development["away_rolling_goals"]
)

equal_features = list(baseline_features) + [
    "home_goals_conceded_5",
    "away_goals_conceded_5",
    "home_opponent_elo_5",
    "away_opponent_elo_5",
]

weighted_features = list(baseline_features) + [
    "home_conceded_5_365d",
    "away_conceded_5_365d",
    "home_opponent_elo_5_365d",
    "away_opponent_elo_5_365d",
]

combined_feature_sets = {
    "equal": equal_features,
    "equal_scoring": equal_features + ["scoring_level"],
    "365d": weighted_features,
    "365d_scoring": weighted_features + ["scoring_level"],
}

required_columns = {
    column
    for columns in combined_feature_sets.values()
    for column in columns
}

missing = sorted(required_columns - set(combined_development.columns))
if missing:
    raise ValueError(
        f"Missing features: {missing}. "
        "Run the relevant feature-generation cells first."
    )

# Restrict this comparison to the two intended models.
combined_templates = {
    name: model_templates[name]
    for name in ["LR", "XGBoost"]
}

combined_folds = list(folds)

for fold_number, (train_idx, valid_idx) in enumerate(
    combined_folds, start=1
):
    train_dates = pd.to_datetime(
        combined_development.iloc[train_idx]["date"]
    )
    valid_dates = pd.to_datetime(
        combined_development.iloc[valid_idx]["date"]
    )

    assert train_dates.max() < valid_dates.min(), (
        f"Fold {fold_number} is not strictly chronological."
    )

In [18]:
# recency experiment with scoring level
combined_results = compare_feature_sets(
    combined_development,
    combined_folds,
    combined_feature_sets,
    combined_templates,
)

combined_fold_comparison = combined_results.pivot(
    index=["model", "fold"],
    columns="feature_set",
    values="log_loss",
).reindex(columns=list(combined_feature_sets))

combined_summary = (
    combined_results
    .groupby(["model", "feature_set"])["log_loss"]
    .mean()
    .unstack("feature_set")
    .reindex(columns=list(combined_feature_sets))
)

print("Per-fold log loss:")
display(combined_fold_comparison.round(6))

print("Mean log loss:")
display(combined_summary.round(6))

Per-fold log loss:


feature_set      equal  equal_scoring      365d  365d_scoring
model   fold                                                 
LR      1     0.852920       0.852064  0.853956      0.853008
        2     0.868797       0.868471  0.870254      0.869888
        3     0.899183       0.899641  0.898189      0.898712
        4     0.868673       0.868382  0.868419      0.868018
XGBoost 1     0.856671       0.855892  0.857802      0.857090
        2     0.870384       0.869935  0.867507      0.867403
        3     0.899103       0.897919  0.896983      0.897554
        4     0.867930       0.868317  0.868216      0.867434

Mean log loss:


feature_set,equal,equal_scoring,365d,365d_scoring
model,,,,
LR,0.872393,0.872140,0.872705,0.872407
XGBoost,0.873522,0.873016,0.872627,0.872370


In [19]:
combined_deltas = pd.DataFrame(
    index=combined_fold_comparison.index
)

combined_deltas["add_scoring_to_equal"] = (
    combined_fold_comparison["equal_scoring"]
    - combined_fold_comparison["equal"]
)

combined_deltas["add_scoring_to_365d"] = (
    combined_fold_comparison["365d_scoring"]
    - combined_fold_comparison["365d"]
)

combined_deltas["add_recency_with_scoring"] = (
    combined_fold_comparison["365d_scoring"]
    - combined_fold_comparison["equal_scoring"]
)

combined_delta_summary = (
    combined_deltas
    .reset_index()
    .melt(
        id_vars=["model", "fold"],
        var_name="comparison",
        value_name="delta",
    )
    .groupby(["model", "comparison"])
    .agg(
        mean_delta=("delta", "mean"),
        folds_improved=("delta", lambda values: (values < -1e-12).sum()),
    )
)

display(combined_deltas.round(6))
display(combined_delta_summary.round(6))

add_scoring_to_equal  add_scoring_to_365d  \
model   fold                                              
LR      1                -0.000856            -0.000948   
        2                -0.000326            -0.000366   
        3                 0.000458             0.000523   
        4                -0.000291            -0.000402   
XGBoost 1                -0.000778            -0.000712   
        2                -0.000449            -0.000103   
        3                -0.001184             0.000572   
        4                 0.000388            -0.000783   

              add_recency_with_scoring  
model   fold                            
LR      1                     0.000944  
        2                     0.001417  
        3                    -0.000928  
        4                    -0.000364  
XGBoost 1                     0.001198  
        2                    -0.002531  
        3                    -0.000365  
        4                    -0.000884

mean_delta  folds_improved
model   comparison                                          
LR      add_recency_with_scoring    0.000267               2
        add_scoring_to_365d        -0.000298               3
        add_scoring_to_equal       -0.000254               3
XGBoost add_recency_with_scoring   -0.000646               3
        add_scoring_to_365d        -0.000257               3
        add_scoring_to_equal       -0.000506               3

In [20]:
# Preparing data and configurations for tuning the models
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import log_loss

# Preserve your existing experiment variables.
tuning_development = (
    matches.loc[pd.to_datetime(matches["date"]) < "2018-01-01"]
    .copy()
    .reset_index(drop=True)
)

# Confirm existing fold indices identify the same matches.
identity_columns = ["date", "home_team", "away_team"]

pd.testing.assert_frame_equal(
    tuning_development[identity_columns],
    development.reset_index(drop=True)[identity_columns],
)

tuning_development["scoring_level"] = (
    tuning_development["home_rolling_goals"]
    + tuning_development["away_rolling_goals"]
)

tuning_features = {
    "LR": list(baseline_features) + [
        "home_goals_conceded_5",
        "away_goals_conceded_5",
        "home_opponent_elo_5",
        "away_opponent_elo_5",
        "scoring_level",
    ],
    "XGBoost": list(baseline_features) + [
        "home_conceded_5_365d",
        "away_conceded_5_365d",
        "home_opponent_elo_5_365d",
        "away_opponent_elo_5_365d",
        "scoring_level",
    ],
}

required = {
    column
    for columns in tuning_features.values()
    for column in columns
}

missing = sorted(required - set(tuning_development.columns))
if missing:
    raise ValueError(f"Missing generated features: {missing}")

tuning_folds = list(folds)

for fold_number, (train_idx, valid_idx) in enumerate(
    tuning_folds, start=1
):
    train_dates = pd.to_datetime(
        tuning_development.iloc[train_idx]["date"]
    )
    valid_dates = pd.to_datetime(
        tuning_development.iloc[valid_idx]["date"]
    )
    assert train_dates.max() < valid_dates.min(), (
        f"Fold {fold_number} is not strictly chronological."
    )

y_tuning = tuning_development["target"]

In [21]:
# Defining bounded parameter searches for tuning
lr_template = clone(model_templates["LR"])
xgb_template = clone(model_templates["XGBoost"])

# Find C even when LogisticRegression is inside a Pipeline.
lr_parameters = lr_template.get_params(deep=True)
c_keys = [
    key
    for key in lr_parameters
    if key == "C" or key.endswith("__C")
]

if len(c_keys) != 1:
    raise ValueError(
        f"Expected one LR regularization parameter; found {c_keys}"
    )

lr_c_key = c_keys[0]
original_c = float(lr_parameters[lr_c_key])

# Include the original setting in the search.
lr_grid = {
    f"model__{lr_c_key}": sorted({
        0.01, 0.1, 1.0, 10.0, 100.0, original_c
    })
}

xgb_parameters = xgb_template.get_params()

xgb_grid = {
    "model__max_depth": [2, 3, 4],
    "model__min_child_weight": [1, 5, 10],
    "model__reg_lambda": [1.0, 5.0, 10.0],
}

# Ensure the original XGBoost configuration is included.
for parameter in ["max_depth", "min_child_weight", "reg_lambda"]:
    original_value = xgb_parameters[parameter]
    grid_key = f"model__{parameter}"

    if original_value is not None:
        xgb_grid[grid_key] = sorted(
            set(xgb_grid[grid_key] + [original_value])
        )

tuning_templates = {
    "LR": lr_template,
    "XGBoost": xgb_template,
}

tuning_grids = {
    "LR": lr_grid,
    "XGBoost": xgb_grid,
}

In [22]:
tuning_searches = {}
tuning_tables = {}
tuning_fold_records = []

for model_name, template in tuning_templates.items():
    print(f"\nTuning {model_name}")

    X_tuning = tuning_development[
        tuning_features[model_name]
    ].astype(float)

    pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median", add_indicator=True),
        ),
        ("model", clone(template)),
    ])

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=tuning_grids[model_name],
        scoring="neg_log_loss",
        cv=tuning_folds,
        refit=True,
        n_jobs=1,
        error_score="raise",
        verbose=1,
    )

    search.fit(X_tuning, y_tuning)
    tuning_searches[model_name] = search

    results = pd.DataFrame(search.cv_results_)
    results["mean_log_loss"] = -results["mean_test_score"]

    parameter_columns = [
        column for column in results.columns
        if column.startswith("param_")
    ]

    tuning_tables[model_name] = (
        results[
            parameter_columns
            + ["mean_log_loss", "rank_test_score"]
        ]
        .sort_values("mean_log_loss")
        .reset_index(drop=True)
    )

    print("Best parameters:", search.best_params_)
    print(f"Best development CV loss: {-search.best_score_:.6f}")
    display(tuning_tables[model_name].head(10))

    # Best candidate's fold losses are already available from the search.
    for split_number, (train_idx, valid_idx) in enumerate(tuning_folds):
        original = clone(pipeline)
        original.fit(
            X_tuning.iloc[train_idx],
            y_tuning.iloc[train_idx],
        )

        np.testing.assert_array_equal(original.classes_, [0, 1, 2])

        original_loss = log_loss(
            y_tuning.iloc[valid_idx],
            original.predict_proba(X_tuning.iloc[valid_idx]),
            labels=[0, 1, 2],
        )

        tuned_loss = -search.cv_results_[
            f"split{split_number}_test_score"
        ][search.best_index_]

        tuning_fold_records.append({
            "model": model_name,
            "fold": split_number + 1,
            "original_log_loss": original_loss,
            "tuned_log_loss": tuned_loss,
            "delta": tuned_loss - original_loss,
        })

tuning_fold_comparison = pd.DataFrame(tuning_fold_records)

tuning_summary = (
    tuning_fold_comparison.groupby("model")
    .agg(
        original_mean_loss=("original_log_loss", "mean"),
        tuned_mean_loss=("tuned_log_loss", "mean"),
        mean_delta=("delta", "mean"),
        folds_improved=("delta", lambda x: (x < -1e-12).sum()),
    )
    .sort_values("tuned_mean_loss")
)

display(tuning_fold_comparison.round(6))
display(tuning_summary.round(6))


Tuning LR
Fitting 4 folds for each of 5 candidates, totalling 20 fits
Best parameters: {'model__model__C': 0.01}
Best development CV loss: 0.872140


,param_model__model__C,mean_log_loss,rank_test_score
0,0.01,0.872140,1
1,0.10,0.872256,2
2,10.00,0.872271,3
3,100.00,0.872271,4
4,1.00,0.872280,5



Tuning XGBoost
Fitting 4 folds for each of 27 candidates, totalling 108 fits
Best parameters: {'model__max_depth': 3, 'model__min_child_weight': 1, 'model__reg_lambda': 1.0}
Best development CV loss: 0.872370


,param_model__max_depth,param_model__min_child_weight,param_model__reg_lambda,mean_log_loss,rank_test_score
0,3,1,1.0,0.872370,1
1,3,10,1.0,0.872431,2
2,3,10,10.0,0.872487,3
3,3,5,1.0,0.872497,4
4,3,5,10.0,0.872505,5
5,3,1,10.0,0.872594,6
6,3,10,5.0,0.872623,7
7,3,5,5.0,0.872709,8
8,3,1,5.0,0.872751,9
9,2,10,1.0,0.873436,10


,model,fold,original_log_loss,tuned_log_loss,delta
0,LR,1,0.852064,0.852064,0.0
1,LR,2,0.868471,0.868471,0.0
2,LR,3,0.899641,0.899641,0.0
3,LR,4,0.868382,0.868382,0.0
4,XGBoost,1,0.857090,0.857090,0.0
5,XGBoost,2,0.867403,0.867403,0.0
6,XGBoost,3,0.897554,0.897554,0.0
7,XGBoost,4,0.867434,0.867434,0.0


,original_mean_loss,tuned_mean_loss,mean_delta,folds_improved
model,,,,
LR,0.87214,0.87214,0.0,0
XGBoost,0.87237,0.87237,0.0,0


In [23]:
#Prepare calibration and selection data
import numpy as np
import pandas as pd

from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator
from sklearn.metrics import log_loss

v2_evaluation_data = matches.copy()
v2_evaluation_data["date"] = pd.to_datetime(
    v2_evaluation_data["date"]
)

v2_evaluation_data["scoring_level"] = (
    v2_evaluation_data["home_rolling_goals"]
    + v2_evaluation_data["away_rolling_goals"]
)

v2_calibration = v2_evaluation_data.loc[
    (v2_evaluation_data["date"] >= "2018-01-01")
    & (v2_evaluation_data["date"] < "2020-01-01")
].copy()

v2_selection = v2_evaluation_data.loc[
    (v2_evaluation_data["date"] >= "2020-01-01")
    & (v2_evaluation_data["date"] < "2022-01-01")
].copy()

assert not v2_calibration.empty
assert not v2_selection.empty

assert (
    pd.to_datetime(tuning_development["date"]).max()
    < v2_calibration["date"].min()
)
assert (
    v2_calibration["date"].max()
    < v2_selection["date"].min()
)

required_columns = {
    column
    for columns in tuning_features.values()
    for column in columns
}

missing = sorted(
    required_columns - set(v2_evaluation_data.columns)
)
if missing:
    raise ValueError(f"Missing generated features: {missing}")

for name, frame in [
    ("Calibration", v2_calibration),
    ("Selection", v2_selection),
]:
    assert set(frame["target"].unique()) == {0, 1, 2}
    print(
        f"{name}: {len(frame)} matches, "
        f"{frame['date'].min().date()} to "
        f"{frame['date'].max().date()}"
    )

Calibration: 1398 matches, 2018-01-02 to 2019-12-18
Selection: 1143 matches, 2020-09-03 to 2021-12-29


In [24]:
# Fit sigmoid calibration and evaluate all four candidates
v2_candidates = {}
v2_selection_probabilities = {}
v2_selection_records = []

for model_name in ["LR", "XGBoost"]:
    columns = list(tuning_features[model_name])

    # Already fitted on all development data by GridSearchCV.
    raw_model = tuning_searches[model_name].best_estimator_

    # Verify the feature order matches the fitted pipeline.
    if hasattr(raw_model, "feature_names_in_"):
        assert list(raw_model.feature_names_in_) == columns

    X_calibration = v2_calibration[columns].astype(float)
    y_calibration = v2_calibration["target"]

    X_selection = v2_selection[columns].astype(float)
    y_selection = v2_selection["target"]

    sigmoid_model = CalibratedClassifierCV(
        estimator=FrozenEstimator(raw_model),
        method="sigmoid",
        ensemble=False,
    )

    # Only calibration-period labels are used here.
    sigmoid_model.fit(X_calibration, y_calibration)

    for variant, estimator in [
        ("raw", raw_model),
        ("sigmoid", sigmoid_model),
    ]:
        candidate_name = f"{model_name} {variant}"

        np.testing.assert_array_equal(
            estimator.classes_, [0, 1, 2]
        )

        probabilities = estimator.predict_proba(X_selection)

        assert np.isfinite(probabilities).all()
        assert ((probabilities >= 0) & (probabilities <= 1)).all()
        np.testing.assert_allclose(
            probabilities.sum(axis=1),
            1.0,
            atol=1e-6,
        )

        v2_candidates[candidate_name] = {
            "model": estimator,
            "features": columns,
        }

        v2_selection_probabilities[candidate_name] = probabilities

        v2_selection_records.append({
            "candidate": candidate_name,
            "selection_matches": len(y_selection),
            "selection_log_loss": log_loss(
                y_selection,
                probabilities,
                labels=[0, 1, 2],
            ),
        })

v2_selection_results = (
    pd.DataFrame(v2_selection_records)
    .sort_values("selection_log_loss")
    .reset_index(drop=True)
)

display(v2_selection_results.round(6))

,candidate,selection_matches,selection_log_loss
0,LR raw,1143,0.834636
1,XGBoost raw,1143,0.834867
2,XGBoost sigmoid,1143,0.834901
3,LR sigmoid,1143,0.836496


In [25]:
# Record selected candidate
v2_selected_name = v2_selection_results.loc[0, "candidate"]
v2_selected_model = v2_candidates[v2_selected_name]["model"]
v2_selected_features = list(
    v2_candidates[v2_selected_name]["features"]
)

print("Selected candidate:", v2_selected_name)
print("Features:", v2_selected_features)

Selected candidate: LR raw
Features: ['elo_difference', 'is_home_team_host', 'neutral', 'form_difference', 'abs_elo_difference', 'home_goals_conceded_5', 'away_goals_conceded_5', 'home_opponent_elo_5', 'away_opponent_elo_5', 'scoring_level']


In [ ]:
# Freeze (save) the selected model (version 2), evaluation checkpoint
from pathlib import Path
from datetime import datetime
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import log_loss, accuracy_score

# Locate the project without a machine-specific home directory.
project_root = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "app.py").is_file()
     and (folder / "results.csv").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Open this notebook from the project folder.")

v2_save_dir = (
    project_root
    / "checkpoints"
    / f"v2_frozen_{datetime.now():%Y%m%d_%H%M%S_%f}"
)
v2_save_dir.mkdir(parents=True, exist_ok=False)

v2_frozen_bundle = {
    "model": v2_selected_model,
    "features": list(v2_selected_features),
    "model_name": v2_selected_name,
    "version": "v2-evaluation",
    "class_names": {0: "Away", 1: "Draw", 2: "Home"},
    "target_definition": (
        "Recorded match outcome including extra time where played; "
        "excluding penalty shootouts."
    ),
    "development_end_exclusive": "2018-01-01",
    "calibration_period": ["2018-01-01", "2020-01-01"],
    "selection_period": ["2020-01-01", "2022-01-01"],
    "calibration_applied": v2_selected_name.endswith("sigmoid"),
    "selection_results": v2_selection_results.copy(),
    "feature_configuration": (
        "Baseline + equal-weight last-five-match defense and "
        "opponent Elo context + scoring level"
    ),
}

joblib.dump(
    v2_frozen_bundle,
    v2_save_dir / "model_bundle.joblib",
)

v2_selection_results.to_csv(
    v2_save_dir / "selection_results.csv",
    index=False,
)

print("Frozen model saved to:", v2_save_dir.relative_to(project_root))

In [27]:
# Compare against corrected v1.1 on identical holdout matches
v1_checkpoint_dir = (
    project_root
    / "checkpoints"
    / "v1_1_corrected_20260915_190534_952693"
)

v1_bundle = joblib.load(
    v1_checkpoint_dir / "model_bundle.joblib"
)
v1_experiment = joblib.load(
    v1_checkpoint_dir / "experiment_data.joblib"
)

# Reload the frozen V2 model to evaluate the actual saved artifact.
v2_bundle = joblib.load(v2_save_dir / "model_bundle.joblib")

v1_holdout = (
    v1_experiment["holdout"]
    .copy()
    .reset_index(drop=True)
)
v1_holdout["date"] = pd.to_datetime(v1_holdout["date"])

v2_holdout = matches.copy()
v2_holdout["date"] = pd.to_datetime(v2_holdout["date"])
v2_holdout = (
    v2_holdout.loc[v2_holdout["date"] >= "2022-01-01"]
    .copy()
    .reset_index(drop=True)
)

v2_holdout["scoring_level"] = (
    v2_holdout["home_rolling_goals"]
    + v2_holdout["away_rolling_goals"]
)

# Stop if the datasets differ: do not compare different match samples.
identity_columns = [
    "date", "home_team", "away_team", "target"
]

pd.testing.assert_frame_equal(
    v1_holdout[identity_columns],
    v2_holdout[identity_columns],
    check_dtype=False,
)

y_holdout = v2_holdout["target"].to_numpy()

evaluation_candidates = {
    "v1.1 evaluation": (
        v1_bundle["model"],
        v1_holdout[v1_bundle["features"]].astype(float),
    ),
    f"v2 {v2_bundle['model_name']}": (
        v2_bundle["model"],
        v2_holdout[v2_bundle["features"]].astype(float),
    ),
}

# Same development-frequency reference for both models.
reference_frequencies = (
    v1_experiment["development"]["target"]
    .value_counts(normalize=True)
    .reindex([0, 1, 2], fill_value=0)
    .to_numpy()
)

reference_probabilities = np.tile(
    reference_frequencies,
    (len(y_holdout), 1),
)

reference_loss = log_loss(
    y_holdout, reference_probabilities, labels=[0, 1, 2]
)

v2_holdout_records = []
v2_holdout_predictions = v2_holdout[identity_columns].copy()

for name, (estimator, X_holdout) in evaluation_candidates.items():
    np.testing.assert_array_equal(estimator.classes_, [0, 1, 2])

    probabilities = estimator.predict_proba(X_holdout)

    assert np.isfinite(probabilities).all()
    assert ((probabilities >= 0) & (probabilities <= 1)).all()
    np.testing.assert_allclose(
        probabilities.sum(axis=1), 1.0, atol=1e-6
    )

    model_loss = log_loss(
        y_holdout, probabilities, labels=[0, 1, 2]
    )

    v2_holdout_records.append({
        "model": name,
        "matches": len(y_holdout),
        "log_loss": model_loss,
        "reference_log_loss": reference_loss,
        "improvement_vs_reference": reference_loss - model_loss,
        "accuracy": accuracy_score(
            y_holdout,
            estimator.classes_[probabilities.argmax(axis=1)],
        ),
    })

    for column_index, outcome in enumerate(["away", "draw", "home"]):
        v2_holdout_predictions[f"{name}_{outcome}"] = (
            probabilities[:, column_index]
        )

v2_holdout_comparison = pd.DataFrame(v2_holdout_records)

v1_loss = v2_holdout_comparison.loc[0, "log_loss"]
v2_holdout_comparison["delta_vs_v1"] = (
    v2_holdout_comparison["log_loss"] - v1_loss
)

display(v2_holdout_comparison.round(6))

,model,matches,log_loss,reference_log_loss,improvement_vs_reference,accuracy,delta_vs_v1
0,v1.1 evaluation,3367,0.886572,1.056874,0.170302,0.602614,0.000000
1,v2 LR raw,3367,0.870196,1.056874,0.186678,0.603505,-0.016376


In [ ]:
v2_holdout_comparison.to_csv(
    v2_save_dir / "holdout_comparison.csv",
    index=False,
)

v2_holdout_predictions.to_csv(
    v2_save_dir / "holdout_predictions.csv",
    index=False,
)

(v2_save_dir / "NEXT_STEPS.txt").write_text(
    "V2 candidate frozen and retrospectively evaluated against v1.1.\n"
    "Deployment artifacts have not been changed.\n"
    "Next: review results and diagnostics, then implement matching "
    "V2 application feature/state logic if proceeding to deployment.\n"
    "The historical holdout was previously inspected for V1; "
    "this is not a fresh independent test.\n",
    encoding="utf-8",
)

print("Model and evaluation saved to:", v2_save_dir.relative_to(project_root))

In [ ]:
# v2_holdout_comparison.to_csv(
#     v2_save_dir / "holdout_comparison.csv",
#     index=False,
# )

# v2_holdout_predictions.to_csv(
#     v2_save_dir / "holdout_predictions.csv",
#     index=False,
# )

# (v2_save_dir / "NEXT_STEPS.txt").write_text(
#     "V2 candidate frozen and retrospectively evaluated against v1.1.\n"
#     "Deployment artifacts have not been changed.\n"
#     "Next: review results and diagnostics, then implement matching "
#     "V2 application feature/state logic if proceeding to deployment.\n"
#     "The historical holdout was previously inspected for V1; "
#     "this is not a fresh independent test.\n",
#     encoding="utf-8",
# )

# print("Model and evaluation saved to:", v2_save_dir.relative_to(project_root))

In [ ]:
# Reload saved evaluation
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import log_loss

# Locate the project without a machine-specific home directory.
project_root = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "app.py").is_file()
     and (folder / "results.csv").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Open this notebook from the project folder.")

completed_checkpoints = sorted(
    path
    for path in (project_root / "checkpoints").glob("v2_frozen_*")
    if (path / "holdout_predictions.csv").exists()
    and (path / "holdout_comparison.csv").exists()
    and (path / "model_bundle.joblib").exists()
)

if not completed_checkpoints:
    raise FileNotFoundError("No completed v2 evaluation checkpoint found.")

review_dir = completed_checkpoints[-1]
print("Reviewing:", review_dir.relative_to(project_root))

frozen_bundle = joblib.load(review_dir / "model_bundle.joblib")

predictions = pd.read_csv(
    review_dir / "holdout_predictions.csv",
    parse_dates=["date"],
)

saved_comparison = pd.read_csv(
    review_dir / "holdout_comparison.csv"
)

print("Frozen candidate:", frozen_bundle["model_name"])
display(saved_comparison)

In [37]:
# Add venue and tournament information
v1_checkpoint_dir = (
    project_root
    / "checkpoints"
    / "v1_1_corrected_20260915_190534_952693"
)

original_experiment = joblib.load(
    v1_checkpoint_dir / "experiment_data.joblib"
)

original_holdout = (
    original_experiment["holdout"]
    .copy()
    .reset_index(drop=True)
)
original_holdout["date"] = pd.to_datetime(original_holdout["date"])

identity_columns = ["date", "home_team", "away_team", "target"]

pd.testing.assert_frame_equal(
    predictions[identity_columns].reset_index(drop=True),
    original_holdout[identity_columns],
    check_dtype=False,
)

review = predictions.copy()

# Handle booleans, 0/1, or their string representations explicitly.
neutral = (
    original_holdout["neutral"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({"true": True, "false": False, "1": True, "0": False,
          "1.0": True, "0.0": False})
)
if neutral.isna().any():
    raise ValueError("Unexpected values in the neutral column.")

review["neutral"] = neutral.to_numpy()
review["tournament"] = original_holdout["tournament"].to_numpy()
review["year"] = review["date"].dt.year

v1_name = "v1.1 evaluation"
v2_name = f"v2 {frozen_bundle['model_name']}"

probability_columns = {
    name: [f"{name}_{outcome}" for outcome in ["away", "draw", "home"]]
    for name in [v1_name, v2_name]
}

for name, columns in probability_columns.items():
    p = review[columns].to_numpy(dtype=float)
    assert np.isfinite(p).all()
    assert ((p >= 0) & (p <= 1)).all()
    np.testing.assert_allclose(p.sum(axis=1), 1.0, atol=1e-6)

# Confirm the saved probabilities reproduce the saved overall scores.
for name, columns in probability_columns.items():
    reproduced = log_loss(
        review["target"], review[columns], labels=[0, 1, 2]
    )
    recorded = saved_comparison.loc[
        saved_comparison["model"] == name, "log_loss"
    ].iloc[0]
    np.testing.assert_allclose(reproduced, recorded, atol=1e-10)

print("Saved predictions and holdout rows verified.")

Saved predictions and holdout rows verified.


In [38]:
# Compare years and match types
def compare_segment(label, mask):
    subset = review.loc[mask]

    if subset.empty:
        return {
            "segment": label,
            "matches": 0,
            "v1_log_loss": np.nan,
            "v2_log_loss": np.nan,
            "delta_v2_minus_v1": np.nan,
        }

    losses = {
        name: log_loss(
            subset["target"],
            subset[columns],
            labels=[0, 1, 2],
        )
        for name, columns in probability_columns.items()
    }

    return {
        "segment": label,
        "matches": len(subset),
        "v1_log_loss": losses[v1_name],
        "v2_log_loss": losses[v2_name],
        "delta_v2_minus_v1": losses[v2_name] - losses[v1_name],
    }


segment_masks = {
    "All matches": pd.Series(True, index=review.index),
    "Neutral": review["neutral"],
    "Non-neutral": ~review["neutral"],
    "World Cup finals": review["tournament"].eq("FIFA World Cup"),
}

segment_review = pd.DataFrame([
    compare_segment(label, mask)
    for label, mask in segment_masks.items()
])

year_review = pd.DataFrame([
    compare_segment(str(year), review["year"].eq(year))
    for year in sorted(review["year"].unique())
])

display(segment_review.round(6))
display(year_review.round(6))

,segment,matches,v1_log_loss,v2_log_loss,delta_v2_minus_v1
0,All matches,3367,0.886572,0.870196,-0.016376
1,Neutral,1205,0.954683,0.915620,-0.039063
2,Non-neutral,2162,0.848610,0.844879,-0.003731
3,World Cup finals,136,1.048881,1.025665,-0.023216


,segment,matches,v1_log_loss,v2_log_loss,delta_v2_minus_v1
0,2022,621,0.957238,0.946070,-0.011168
1,2023,802,0.845379,0.832982,-0.012396
2,2024,988,0.917510,0.897195,-0.020315
3,2025,781,0.816346,0.799105,-0.017242
4,2026,175,0.963336,0.936345,-0.026991


In [39]:
# Inspect probabiolity reliability for each outcome
reliability_parts = []

for class_id, outcome in enumerate(["away", "draw", "home"]):
    probabilities = review[f"{v2_name}_{outcome}"].to_numpy()

    outcome_data = pd.DataFrame({
        "probability": probabilities,
        "occurred": (
            review["target"].to_numpy() == class_id
        ).astype(int),
    })

    outcome_data["bin"] = pd.cut(
        outcome_data["probability"],
        bins=np.linspace(0, 1, 11),
        include_lowest=True,
    )

    table = (
        outcome_data.groupby("bin", observed=True)
        .agg(
            mean_probability=("probability", "mean"),
            observed_frequency=("occurred", "mean"),
            matches=("occurred", "size"),
        )
        .reset_index()
    )

    table["observed_minus_predicted"] = (
        table["observed_frequency"] - table["mean_probability"]
    )
    table.insert(0, "outcome", outcome)

    reliability_parts.append(table)

    print(f"\n{outcome.capitalize()}")
    display(table.round(4))

reliability_review = pd.concat(
    reliability_parts, ignore_index=True
)


Away


,outcome,bin,mean_probability,observed_frequency,matches,observed_minus_predicted
0,away,"(-0.001, 0.1]",0.0514,0.0551,871,0.0037
1,away,"(0.1, 0.2]",0.1491,0.1938,676,0.0447
2,away,"(0.2, 0.3]",0.2473,0.2824,510,0.0350
3,away,"(0.3, 0.4]",0.3496,0.3852,366,0.0357
4,away,"(0.4, 0.5]",0.4494,0.4270,274,-0.0224
5,away,"(0.5, 0.6]",0.5480,0.5756,238,0.0276
6,away,"(0.6, 0.7]",0.6460,0.6343,175,-0.0117
7,away,"(0.7, 0.8]",0.7477,0.7801,141,0.0325
8,away,"(0.8, 0.9]",0.8532,0.9241,79,0.0709
9,away,"(0.9, 1.0]",0.9326,0.9730,37,0.0403



Draw


,outcome,bin,mean_probability,observed_frequency,matches,observed_minus_predicted
0,draw,"(-0.001, 0.1]",0.0684,0.0670,358,-0.0014
1,draw,"(0.1, 0.2]",0.1553,0.1683,1004,0.0130
2,draw,"(0.2, 0.3]",0.2514,0.2693,1686,0.0179
3,draw,"(0.3, 0.4]",0.3188,0.3197,319,0.0009



Home


,outcome,bin,mean_probability,observed_frequency,matches,observed_minus_predicted
0,home,"(-0.001, 0.1]",0.0542,0.0536,224,-0.0006
1,home,"(0.1, 0.2]",0.1510,0.1329,316,-0.0181
2,home,"(0.2, 0.3]",0.2507,0.2126,334,-0.0381
3,home,"(0.3, 0.4]",0.3512,0.3270,367,-0.0242
4,home,"(0.4, 0.5]",0.4507,0.4103,407,-0.0404
5,home,"(0.5, 0.6]",0.5500,0.4867,413,-0.0633
6,home,"(0.6, 0.7]",0.6513,0.5660,394,-0.0853
7,home,"(0.7, 0.8]",0.7510,0.7125,400,-0.0385
8,home,"(0.8, 0.9]",0.8470,0.8472,360,0.0002
9,home,"(0.9, 1.0]",0.9369,0.9474,152,0.0105


In [ ]:
# Save frozen diagnostics, and extract the 
segment_review.to_csv(
    review_dir / "diagnostics_segments.csv", index=False
)
year_review.to_csv(
    review_dir / "diagnostics_years.csv", index=False
)
reliability_review.to_csv(
    review_dir / "diagnostics_reliability.csv", index=False
)

print("Diagnostics saved to:", review_dir.relative_to(project_root))


In [30]:
# Preparing features for lightGBM testing
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import log_loss
from lightgbm import LGBMClassifier

# Separate frame: preserve the existing experiment variables.
lgb_development = (
    matches.loc[pd.to_datetime(matches["date"]) < "2018-01-01"]
    .copy()
    .reset_index(drop=True)
)

# Confirm the saved fold indices still refer to the same matches.
identity_columns = ["date", "home_team", "away_team"]

pd.testing.assert_frame_equal(
    lgb_development[identity_columns],
    development.reset_index(drop=True)[identity_columns],
)

lgb_development["scoring_level"] = (
    lgb_development["home_rolling_goals"]
    + lgb_development["away_rolling_goals"]
)

base = list(baseline_features)

defense = [
    "home_goals_conceded_5",
    "away_goals_conceded_5",
]

opponent_context = [
    "home_opponent_elo_5",
    "away_opponent_elo_5",
]

lgb_feature_sets = {
    "baseline": base,
    "plus_defense": base + defense,
    "plus_context_equal": base + defense + opponent_context,
    "plus_context_scoring_equal": (
        base + defense + opponent_context + ["scoring_level"]
    ),
}

# Weighted columns replace the equal-weight defense/context columns.
for suffix in ["180d", "365d"]:
    weighted_columns = [
        f"home_conceded_5_{suffix}",
        f"away_conceded_5_{suffix}",
        f"home_opponent_elo_5_{suffix}",
        f"away_opponent_elo_5_{suffix}",
    ]

    lgb_feature_sets[f"plus_context_{suffix}"] = (
        base + weighted_columns
    )
    lgb_feature_sets[f"plus_context_scoring_{suffix}"] = (
        base + weighted_columns + ["scoring_level"]
    )

required_columns = {
    column
    for columns in lgb_feature_sets.values()
    for column in columns
}

missing_columns = sorted(required_columns - set(lgb_development.columns))

if missing_columns:
    raise ValueError(
        "These features have not been generated in matches yet: "
        f"{missing_columns}. Run the relevant feature-generation cells first."
    )

print(f"Ready: {len(lgb_feature_sets)} feature sets.")

Ready: 8 feature sets.


In [31]:
lgb_template = LGBMClassifier(
    objective="multiclass",
    n_estimators=150,
    learning_rate=0.05,
    max_depth=3,
    num_leaves=8,
    min_child_samples=20,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=1,
    verbosity=-1,
)

lgb_folds = list(folds)
lgb_records = []

for feature_name, columns in lgb_feature_sets.items():
    X = lgb_development[columns].astype(float)
    y = lgb_development["target"]

    for fold_number, (train_idx, valid_idx) in enumerate(
        lgb_folds, start=1
    ):
        # Verify each validation period follows its training period.
        train_dates = pd.to_datetime(
            lgb_development.iloc[train_idx]["date"]
        )
        valid_dates = pd.to_datetime(
            lgb_development.iloc[valid_idx]["date"]
        )

        assert train_dates.max() < valid_dates.min(), (
            f"Fold {fold_number} is not strictly chronological."
        )

        estimator = Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="median", add_indicator=True),
            ),
            ("model", clone(lgb_template)),
        ])

        estimator.fit(X.iloc[train_idx], y.iloc[train_idx])

        np.testing.assert_array_equal(
            estimator.classes_, [0, 1, 2]
        )

        probabilities = estimator.predict_proba(X.iloc[valid_idx])

        loss = log_loss(
            y.iloc[valid_idx],
            probabilities,
            labels=[0, 1, 2],
        )

        lgb_records.append({
            "model": "LightGBM",
            "feature_set": feature_name,
            "fold": fold_number,
            "n_features": len(columns),
            "log_loss": loss,
        })

    print(f"Completed: {feature_name}")

lgb_results = pd.DataFrame(lgb_records)

Completed: baseline
Completed: plus_defense
Completed: plus_context_equal
Completed: plus_context_scoring_equal
Completed: plus_context_180d
Completed: plus_context_scoring_180d
Completed: plus_context_365d
Completed: plus_context_scoring_365d


In [32]:
# Requires lgb_results from the LightGBM fitting cell.
lgb_fold_comparison = lgb_results.pivot(
    index="fold",
    columns="feature_set",
    values="log_loss",
)

display(lgb_fold_comparison.round(6))

feature_set,baseline,plus_context_180d,plus_context_365d,plus_context_equal,plus_context_scoring_180d,plus_context_scoring_365d,plus_context_scoring_equal,plus_defense
fold,,,,,,,,
1,0.865160,0.857845,0.856915,0.855117,0.857489,0.856410,0.853471,0.857169
2,0.874749,0.868880,0.867717,0.871036,0.868674,0.867238,0.869440,0.865915
3,0.912841,0.900443,0.899774,0.900584,0.900027,0.901611,0.900327,0.901894
4,0.875079,0.868464,0.868285,0.870131,0.869572,0.868369,0.869944,0.871027


In [33]:
lgb_incremental = pd.DataFrame({
    "add_defense": (
        lgb_fold_comparison["plus_defense"]
        - lgb_fold_comparison["baseline"]
    ),
    "add_opponent_context": (
        lgb_fold_comparison["plus_context_equal"]
        - lgb_fold_comparison["plus_defense"]
    ),
    "add_scoring_equal": (
        lgb_fold_comparison["plus_context_scoring_equal"]
        - lgb_fold_comparison["plus_context_equal"]
    ),
})

for suffix in ["180d", "365d"]:
    lgb_incremental[f"recency_{suffix}_without_scoring"] = (
        lgb_fold_comparison[f"plus_context_{suffix}"]
        - lgb_fold_comparison["plus_context_equal"]
    )
    lgb_incremental[f"recency_{suffix}_with_scoring"] = (
        lgb_fold_comparison[f"plus_context_scoring_{suffix}"]
        - lgb_fold_comparison["plus_context_scoring_equal"]
    )

lgb_incremental_summary = pd.DataFrame({
    "mean_delta": lgb_incremental.mean(),
    "folds_improved": (lgb_incremental < 0).sum(),
})

display(lgb_incremental.round(6))
display(lgb_incremental_summary.round(6))

,add_defense,add_opponent_context,add_scoring_equal,recency_180d_without_scoring,recency_180d_with_scoring,recency_365d_without_scoring,recency_365d_with_scoring
fold,,,,,,,
1,-0.007991,-0.002052,-0.001646,0.002727,0.004018,0.001798,0.002939
2,-0.008834,0.005120,-0.001596,-0.002155,-0.000766,-0.003319,-0.002202
3,-0.010946,-0.001311,-0.000257,-0.000140,-0.000300,-0.000810,0.001284
4,-0.004052,-0.000895,-0.000188,-0.001668,-0.000372,-0.001846,-0.001574


,mean_delta,folds_improved
add_defense,-0.007956,4
add_opponent_context,0.000216,3
add_scoring_equal,-0.000922,4
recency_180d_without_scoring,-0.000309,3
recency_180d_with_scoring,0.000645,3
recency_365d_without_scoring,-0.001044,3
recency_365d_with_scoring,0.000112,2


In [ ]:
from pathlib import Path
from datetime import datetime
import joblib

# Locate the project without a machine-specific home directory.
project_root = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "app.py").is_file()
     and (folder / "results.csv").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Open this notebook from the project folder.")
save_dir = (
    project_root
    / "checkpoints"
    / f"v2_feature_research_{datetime.now():%Y%m%d_%H%M%S_%f}"
)
save_dir.mkdir(parents=True, exist_ok=False)

variable_names = [
    "matches",
    "development",
    "folds",
    "baseline_features",
    "model_templates",
    "scoring_feature_sets",
    "scoring_results",
    "recency_feature_sets",
    "recency_results",
    "combined_results",
    "combined_feature_sets"
]

snapshot = {
    name: globals()[name]
    for name in variable_names
    if name in globals()
}

joblib.dump(snapshot, save_dir / "research_state.joblib")

print("Saved to:", save_dir.relative_to(project_root))
print("Saved variables:", ", ".join(snapshot))
print(
    "Not available:",
    ", ".join(name for name in variable_names if name not in snapshot)
    or "None",
)

In [35]:
v1_checkpoint_dir = (
    project_root
    / "checkpoints"
    / "v1_1_corrected_20260915_190534_952693"
)

v1_bundle = joblib.load(
    v1_checkpoint_dir / "model_bundle.joblib"
)
v1_experiment = joblib.load(
    v1_checkpoint_dir / "experiment_data.joblib"
)

# Reload the frozen V2 model to evaluate the actual saved artifact.
v2_bundle = joblib.load(v2_save_dir / "model_bundle.joblib")

v1_holdout = (
    v1_experiment["holdout"]
    .copy()
    .reset_index(drop=True)
)
v1_holdout["date"] = pd.to_datetime(v1_holdout["date"])

v2_holdout = matches.copy()
v2_holdout["date"] = pd.to_datetime(v2_holdout["date"])
v2_holdout = (
    v2_holdout.loc[v2_holdout["date"] >= "2022-01-01"]
    .copy()
    .reset_index(drop=True)
)

v2_holdout["scoring_level"] = (
    v2_holdout["home_rolling_goals"]
    + v2_holdout["away_rolling_goals"]
)

# Stop if the datasets differ: do not compare different match samples.
identity_columns = [
    "date", "home_team", "away_team", "target"
]

pd.testing.assert_frame_equal(
    v1_holdout[identity_columns],
    v2_holdout[identity_columns],
    check_dtype=False,
)

y_holdout = v2_holdout["target"].to_numpy()

evaluation_candidates = {
    "v1.1 evaluation": (
        v1_bundle["model"],
        v1_holdout[v1_bundle["features"]].astype(float),
    ),
    f"v2 {v2_bundle['model_name']}": (
        v2_bundle["model"],
        v2_holdout[v2_bundle["features"]].astype(float),
    ),
}

# Same development-frequency reference for both models.
reference_frequencies = (
    v1_experiment["development"]["target"]
    .value_counts(normalize=True)
    .reindex([0, 1, 2], fill_value=0)
    .to_numpy()
)

reference_probabilities = np.tile(
    reference_frequencies,
    (len(y_holdout), 1),
)

reference_loss = log_loss(
    y_holdout, reference_probabilities, labels=[0, 1, 2]
)

v2_holdout_records = []
v2_holdout_predictions = v2_holdout[identity_columns].copy()

for name, (estimator, X_holdout) in evaluation_candidates.items():
    np.testing.assert_array_equal(estimator.classes_, [0, 1, 2])

    probabilities = estimator.predict_proba(X_holdout)

    assert np.isfinite(probabilities).all()
    assert ((probabilities >= 0) & (probabilities <= 1)).all()
    np.testing.assert_allclose(
        probabilities.sum(axis=1), 1.0, atol=1e-6
    )

    model_loss = log_loss(
        y_holdout, probabilities, labels=[0, 1, 2]
    )

    v2_holdout_records.append({
        "model": name,
        "matches": len(y_holdout),
        "log_loss": model_loss,
        "reference_log_loss": reference_loss,
        "improvement_vs_reference": reference_loss - model_loss,
        "accuracy": accuracy_score(
            y_holdout,
            estimator.classes_[probabilities.argmax(axis=1)],
        ),
    })

    for column_index, outcome in enumerate(["away", "draw", "home"]):
        v2_holdout_predictions[f"{name}_{outcome}"] = (
            probabilities[:, column_index]
        )

v2_holdout_comparison = pd.DataFrame(v2_holdout_records)

v1_loss = v2_holdout_comparison.loc[0, "log_loss"]
v2_holdout_comparison["delta_vs_v1"] = (
    v2_holdout_comparison["log_loss"] - v1_loss
)

display(v2_holdout_comparison.round(6))

,model,matches,log_loss,reference_log_loss,improvement_vs_reference,accuracy,delta_vs_v1
0,v1.1 evaluation,3367,0.886572,1.056874,0.170302,0.602614,0.000000
1,v2 LR raw,3367,0.870196,1.056874,0.186678,0.603505,-0.016376
